In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from cs_priors.simulation.mixing_model import quick_frequency_sim
from cs_priors.solvers.freq_lasso import frequency_lasso_solve
from cs_priors.solvers.freq_group_lasso import frequency_group_lasso_solve
from cs_priors.solvers.freq_sparse_group_lasso import (
    frequency_sparse_group_lasso_solve,
)
from cs_priors.solvers.freq_sparse_prior import sparse_prior_solve
from cs_priors.plotting.plot_complex import plot_matrices
from cs_priors.metrics.benchmark import grid_benchmark

sns.set_theme(style="whitegrid")

In [ ]:
def slice_positive_frequencies(sim, F=None):
    keep = np.flatnonzero(sim.freqs > 0)
    if F is not None:
        keep = keep[:F]

    sim.A = sim.A[:, :, keep]
    sim.Y = sim.Y[:, keep]
    sim.X = sim.X[:, keep]
    sim.X_pinv = sim.X_pinv[:, keep]
    sim.Y_clean = sim.Y_clean[:, keep]
    sim.eta = sim.eta[:, keep]
    sim.delta = sim.delta[:, keep]
    sim.freqs = sim.freqs[keep]
    return sim


def plot_active_frequency_magnitudes(sim, title="Active source magnitudes by frequency"):
    positive_mask = sim.freqs > 0
    freqs = sim.freqs[positive_mask]

    fig, ax = plt.subplots(figsize=(10, 4))
    for idx in sim.active_indices:
        ax.plot(freqs, np.abs(sim.X[idx, positive_mask]), label=f"Source {idx}")

    ax.set_title(title)
    ax.set_xlabel("Frequency [Hz]")
    ax.set_ylabel(r"$|X_s(f)|$")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_benchmark_confidence_intervals(
    df,
    x="sensor_snr_db",
    metrics=("precision", "f1"),
    hue="method",
    x_label=None,
    y_lim=(0.0, 1.05),
    figsize=(12, 4),
):
    if x_label is None:
        x_label = x

    fig, axes = plt.subplots(1, len(metrics), figsize=figsize, sharex=True, sharey=True)

    if len(metrics) == 1:
        axes = [axes]

    for ax, metric in zip(axes, metrics):
        sns.lineplot(
            data=df,
            x=x,
            y=metric,
            hue=hue,
            estimator="mean",
            errorbar=("ci", 95),
            marker="o",
            ax=ax,
        )
        ax.set_title(f"{metric.upper()} vs {x}")
        ax.set_xlabel(x_label)
        ax.set_ylabel(metric)
        ax.set_ylim(*y_lim)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
params = {
    "num_mics": 4,
    "num_sources": 10,
    "num_active": 2,
    "seed": 0,
    "sampling_rate": 2000.0,
    "duration": 0.01,
    "source_distance": 0.5,
    "mic_radius": 0.1,
    "array_type": "circular",
    "component_amplitude": 10.0,
    "magnitude_jitter": 0.05,
    "min_freq_hz": 1.0,
    "sensor_snr_db": None,
    "model_snr_db": None,
    "inverse_method": "mp",
}

F = None

sim = quick_frequency_sim(**params)
sim = slice_positive_frequencies(sim, F=F)

plot_matrices(
    [sim.X, sim.X_pinv],
    titles=["X", "X_pinv"],
)

print("Active indices:", sim.active_indices)
print("X shape:", sim.X.shape)
print("Y shape:", sim.Y.shape)
print("A shape:", sim.A.shape)



In [ ]:
X_lasso = frequency_lasso_solve(sim.Y, sim.A, X_start=sim.X_pinv)

X_group_lasso = frequency_group_lasso_solve(
    sim.Y,
    sim.A,
    grouping="frequency",
    X_start=sim.X_pinv,
)

X_sparse_group_lasso = frequency_sparse_group_lasso_solve(
    sim.Y,
    sim.A,
    alpha=1e-4,
    grouping="frequency",
    max_iter=5000,
    seed=0,
    X_start=sim.X_pinv,
)

X_sparse_prior = sparse_prior_solve(
    sim.X_pinv,
    sim.A,
    grouping="frequency",
    precision=1.0,
    eps=0.005,
)

X_sparse_prior_from_true = sparse_prior_solve(
    sim.X,
    sim.A,
    grouping="frequency",
)


# This plot shows LASSO is making the solution worse than the pseudoinverse
plot_matrices(
    [
        sim.X,
        sim.X_pinv,
        X_lasso,
        X_group_lasso,
        X_sparse_group_lasso,
        X_sparse_prior,
        X_sparse_prior_from_true,
    ],
    titles=[
        "True solution X",
        "Pseudoinverse solution",
        "LASSO solution",
        "Group LASSO solution",
        "Sparse Group LASSO solution",
        "Sparse Prior from X_pinv",
        "Sparse Prior from X_true",
    ],
    show_values=True,
    dpi=70,
)

plt.show()
# The metrics 

import pandas as pd

from cs_priors.metrics import source_leakage_metrics, noise_threshold

methods = {
    "True solution X": sim.X,
    "Pseudoinverse solution": sim.X_pinv,
    "LASSO solution": X_lasso,
    "Group LASSO solution": X_group_lasso,
    "Sparse Group LASSO solution": X_sparse_group_lasso,
    "Sparse Prior from X_pinv": X_sparse_prior,
    "Sparse Prior from X_true": X_sparse_prior_from_true,
}

support_tol = noise_threshold(sim.X)
row_floor_ratio = 0.10  # count an inactive source as "visibly active" if it exceeds 10% of the strongest true row

rows = []
for name, X_hat in methods.items():
    m = source_leakage_metrics(
        sim.X,
        X_hat,
        tol=support_tol,
        row_floor_ratio=row_floor_ratio,
    )
    rows.append(
        {
            "method": name,
            "on_support_%": 100 * m["active_energy_ratio"],
            "off_support_%": 100 * m["inactive_energy_ratio"],
            f"inactive_rows_above_{int(100*row_floor_ratio)}pct": m["inactive_rows_above_floor"],
            "max_inactive_row_%_of_true_peak": 100 * m["max_inactive_row_ratio"],
            "inactive_to_active_energy_ratio": m["inactive_to_active_energy_ratio"],
            "relative_error": m["relative_error"],
            "precision": m["precision"],
            "recall": m["recall"],
            "f1": m["f1"],
        }
    )

df_leakage = pd.DataFrame(rows).sort_values("off_support_%")
display(df_leakage.round(3))


In [ ]:
positive_mask = sim.freqs > 0

rows = []
for idx in sim.active_indices:
    magnitudes = np.abs(sim.X[idx, positive_mask])
    rows.append(
        {
            "source": idx,
            "min_abs_X": float(magnitudes.min()),
            "max_abs_X": float(magnitudes.max()),
            "mean_abs_X": float(magnitudes.mean()),
            "std_abs_X": float(magnitudes.std()),
        }
    )

display(pd.DataFrame(rows))

In [ ]:
methods = {
    
    "Freq LASSO": lambda sim: frequency_lasso_solve(
        sim.Y,
        sim.A,
        alpha=1e-4,
        max_iter=10000,
        X_start=sim.X_pinv,
    ),
    "Freq Group LASSO": lambda sim: frequency_group_lasso_solve(
        sim.Y,
        sim.A,
        alpha=1e-4,
        grouping="frequency",
        max_iter=5000,
        seed=0,
    ),
    # "Freq Sparse Group LASSO": lambda sim: frequency_sparse_group_lasso_solve(
    #     sim.Y,
    #     sim.A,
    #     alpha=1e-4,
    #     grouping="frequency",
    #     max_iter=5000,
    #     seed=0,
    # ),
    # "Sparse Prior from X_pinv": lambda sim: sparse_prior_solve(
    #     sim.X_pinv,
    #     sim.A,
    #     grouping="frequency",
    #     precision=1.0,
    #     eps=0.005,
    # ),
    # "Sparse Prior from X_true": lambda sim: sparse_prior_solve(
    #     sim.X,
    #     sim.A,
    #     grouping="frequency",
    # ),
    "Pseudoinverse solution": lambda sim: sim.X_pinv,
}

In [ ]:
param_grid = {
    "num_mics": [10],
    "num_sources": [10],
    "num_active": [2],
    "sampling_rate": [2000.0],
    "duration": [0.05],
    "source_distance": [0.5],
    "mic_radius": [0.1],
    "array_type": ["circular"],
    "component_amplitude": [10.0],
    "magnitude_jitter": [0.05],
    "min_freq_hz": [1.0],
    "sensor_snr_db": [1e-99, 5, 10, 20, 25, 30, 35, 40, 50, 60, 70, 80],
    "model_snr_db": [None],
    "inverse_method": ["ridge"],
}

num_seeds = 5 # 32 extra seconds per seed
n_jobs = -2



In [ ]:
rows = grid_benchmark(
    sim_factory=quick_frequency_sim,
    param_grid=param_grid,
    methods=methods,
    num_seeds=num_seeds,
    tol=None,
    n_jobs=n_jobs,
)

df = pd.DataFrame(rows)

summary = (
    df.groupby(["sensor_snr_db", "method"], as_index=False)[
        ["precision", "f1", "relative_error"]
    ]
    .mean()
    .sort_values(["sensor_snr_db", "method"])
)

display(summary)

In [ ]:
plot_benchmark_confidence_intervals(
    df,
    x="sensor_snr_db",
    metrics=("precision", "f1"),
    x_label="sensor_snr_db (higher = less noise)",
    figsize=(18, 5),

)

In [ ]:
plot_benchmark_confidence_intervals(
    df,
    x="sensor_snr_db",
    metrics=("relative_error",),
    x_label="sensor_snr_db (higher = less noise)",
    y_lim=(-1.0, float(df["relative_error"].max()/2) * 1.05),
    figsize=(7, 4),
)